In [1]:
import pandas as pd

In [8]:
# use the original df to find category scales
df = pd.read_pickle('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/preprocessed_cleaned_all_result.pkl')

In [2]:
# use the title mapper to create titles for the beliefs
title_mapper = pd.read_excel("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/title_feature_match_updated_state_level.xlsx")
title_mapper.head()

,Feature,Question type,Question model,Match Dataset,Match ID,Match ID numeric (this is the V variable code for the ANES),Note from Yuyue,what to do,2024 Match Dataset,2024 Match ID,...,Finish recode,GSS,ANES/CCES,Our dataset,Warning,Target Quality,Some benchmark available,Add area-level R-D as pred. ?,Add area-level matched pre-2024 past score as pred. ?,Area-level pred. size
0,OPINION ON THE NUMBER OF IMMIGRANTS IN AMERICA,Likert,Ordinal,ANES2020,POST: WHAT SHOULD IMMIGRATION LEVELS BE (IMMIG...,V202232,NaN,"Keep 1-5 as coded (1=increased a lot, 5=decrea...",ANES2024,POST: WHAT SHOULD IMMIGRATION LEVELS BE (IMMIG...,...,1.0,NaN,NaN,NaN,NaN,GOOD,True,1.0,1.0,2.0
1,OPINION ON THE STATEMENT << IMMIGRANTS INCREAS...,Likert,Ordinal,ANES2020,CSES5_IMMCRIME or GSS_IMMCRIMEGSS,V202420 or V202600,Here I chose V202420 because there are no actu...,Pick whichever beteween these two is less miss...,ANES2024,CSES6_IMMCRIME,...,1.0,drop neither agree or disagree respondents,drop neither agree or disagree respondents,NaN,NaN,GOOD,True,1.0,1.0,2.0
2,OPINION ON THE STATEMENT << IMMIGRANTS TAKE JO...,Likert,Ordinal,ANES2020,POST: HOW LIKELY IMMIGRATION WILL TAKE AWAY JO...,V202233,NaN,Reverse 1-4 so 4=extremely likely and 1=not at...,ANES2024,POST: HOW LIKELY IMMIGRATION WILL TAKE AWAY JO...,...,1.0,drop the neutral option,NaN,NaN,NaN,GOOD,True,1.0,1.0,2.0
3,OPINION ON WHAT SHOULD BE DONE WITH ILLEGAL IM...,Likert,Ordinal,ANES2020,PRE: U.S. GOVERNMENT POLICY TOWARD UNAUTHORIZE...,V201417,NaN,Collapse to 3 classes: 1=identified/deported; ...,ANES2024,PRE: U.S. GOVERNMENT POLICY TOWARD UNAUTHORIZE...,...,1.0,NaN,ANES Collapse to 3 classes: 1=identified/depor...,NaN,NaN,GOOD,True,1.0,1.0,2.0
4,OPINION ON WHETHER DIVORCE IN THE USA SHOULD B...,Likert,Ordinal,ANES2020,POST: FEWER PROBLEMS IF THERE WAS MORE EMPHASI...,V202265,NaN,Reverse 1-5 so higher=more agreement with trad...,NaN,NaN,...,1.0,"recode so 2 is the right level as it is, 3 is ...",NaN,NaN,"GSS 2024 only exact match, no state-level matc...",NaN,True,1.0,0.0,1.0


In [4]:
# Remove rows where all values are NaN
title_mapper = title_mapper.dropna(how='all')
print(len(title_mapper))
title_mapper.loc[27, 'Feature'] = "increase fossil fuel production"

# Remove rows where "question type" is NaN (case-insensitive column match)
title_mapper = title_mapper[title_mapper['Question type'].notna()].reset_index(drop=True)
print(len(title_mapper))
title_mapper.head()

38
38


,Feature,Question type,Question model,Match Dataset,Match ID,Match ID numeric (this is the V variable code for the ANES),Note from Yuyue,what to do,2024 Match Dataset,2024 Match ID,...,Finish recode,GSS,ANES/CCES,Our dataset,Warning,Target Quality,Some benchmark available,Add area-level R-D as pred. ?,Add area-level matched pre-2024 past score as pred. ?,Area-level pred. size
0,OPINION ON THE NUMBER OF IMMIGRANTS IN AMERICA,Likert,Ordinal,ANES2020,POST: WHAT SHOULD IMMIGRATION LEVELS BE (IMMIG...,V202232,NaN,"Keep 1-5 as coded (1=increased a lot, 5=decrea...",ANES2024,POST: WHAT SHOULD IMMIGRATION LEVELS BE (IMMIG...,...,1.0,NaN,NaN,NaN,NaN,GOOD,True,1.0,1.0,2.0
1,OPINION ON THE STATEMENT << IMMIGRANTS INCREAS...,Likert,Ordinal,ANES2020,CSES5_IMMCRIME or GSS_IMMCRIMEGSS,V202420 or V202600,Here I chose V202420 because there are no actu...,Pick whichever beteween these two is less miss...,ANES2024,CSES6_IMMCRIME,...,1.0,drop neither agree or disagree respondents,drop neither agree or disagree respondents,NaN,NaN,GOOD,True,1.0,1.0,2.0
2,OPINION ON THE STATEMENT << IMMIGRANTS TAKE JO...,Likert,Ordinal,ANES2020,POST: HOW LIKELY IMMIGRATION WILL TAKE AWAY JO...,V202233,NaN,Reverse 1-4 so 4=extremely likely and 1=not at...,ANES2024,POST: HOW LIKELY IMMIGRATION WILL TAKE AWAY JO...,...,1.0,drop the neutral option,NaN,NaN,NaN,GOOD,True,1.0,1.0,2.0
3,OPINION ON WHAT SHOULD BE DONE WITH ILLEGAL IM...,Likert,Ordinal,ANES2020,PRE: U.S. GOVERNMENT POLICY TOWARD UNAUTHORIZE...,V201417,NaN,Collapse to 3 classes: 1=identified/deported; ...,ANES2024,PRE: U.S. GOVERNMENT POLICY TOWARD UNAUTHORIZE...,...,1.0,NaN,ANES Collapse to 3 classes: 1=identified/depor...,NaN,NaN,GOOD,True,1.0,1.0,2.0
4,OPINION ON WHETHER DIVORCE IN THE USA SHOULD B...,Likert,Ordinal,ANES2020,POST: FEWER PROBLEMS IF THERE WAS MORE EMPHASI...,V202265,NaN,Reverse 1-5 so higher=more agreement with trad...,NaN,NaN,...,1.0,"recode so 2 is the right level as it is, 3 is ...",NaN,NaN,"GSS 2024 only exact match, no state-level matc...",NaN,True,1.0,0.0,1.0


In [5]:
title_mapper['cleaned_feature'] = (
    'Belief_'
    + title_mapper['Feature']
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'<<|>>|«|»', '', regex=True)
        .str.replace(r"[''´’`]s\b", '_s', regex=True)  # replace all types of 's with _s
        .str.replace(r"[^\w\s]", "", regex=True)   # remove punctuation like , . ' " etc.
        .str.replace(r'\s+', '_', regex=True)
        .str.replace('us_has_become', 'u_s_has_become', regex=False)
        .str.replace(r'_+', '_', regex=True)
        .str.replace('afferm', 'affirm', regex=False)
        .str.strip('_')
    + '_Symbol'
)

title_mapper.head()

,Feature,Question type,Question model,Match Dataset,Match ID,Match ID numeric (this is the V variable code for the ANES),Note from Yuyue,what to do,2024 Match Dataset,2024 Match ID,...,GSS,ANES/CCES,Our dataset,Warning,Target Quality,Some benchmark available,Add area-level R-D as pred. ?,Add area-level matched pre-2024 past score as pred. ?,Area-level pred. size,cleaned_feature
0,OPINION ON THE NUMBER OF IMMIGRANTS IN AMERICA,Likert,Ordinal,ANES2020,POST: WHAT SHOULD IMMIGRATION LEVELS BE (IMMIG...,V202232,NaN,"Keep 1-5 as coded (1=increased a lot, 5=decrea...",ANES2024,POST: WHAT SHOULD IMMIGRATION LEVELS BE (IMMIG...,...,NaN,NaN,NaN,NaN,GOOD,True,1.0,1.0,2.0,Belief_opinion_on_the_number_of_immigrants_in_...
1,OPINION ON THE STATEMENT << IMMIGRANTS INCREAS...,Likert,Ordinal,ANES2020,CSES5_IMMCRIME or GSS_IMMCRIMEGSS,V202420 or V202600,Here I chose V202420 because there are no actu...,Pick whichever beteween these two is less miss...,ANES2024,CSES6_IMMCRIME,...,drop neither agree or disagree respondents,drop neither agree or disagree respondents,NaN,NaN,GOOD,True,1.0,1.0,2.0,Belief_opinion_on_the_statement_immigrants_inc...
2,OPINION ON THE STATEMENT << IMMIGRANTS TAKE JO...,Likert,Ordinal,ANES2020,POST: HOW LIKELY IMMIGRATION WILL TAKE AWAY JO...,V202233,NaN,Reverse 1-4 so 4=extremely likely and 1=not at...,ANES2024,POST: HOW LIKELY IMMIGRATION WILL TAKE AWAY JO...,...,drop the neutral option,NaN,NaN,NaN,GOOD,True,1.0,1.0,2.0,Belief_opinion_on_the_statement_immigrants_tak...
3,OPINION ON WHAT SHOULD BE DONE WITH ILLEGAL IM...,Likert,Ordinal,ANES2020,PRE: U.S. GOVERNMENT POLICY TOWARD UNAUTHORIZE...,V201417,NaN,Collapse to 3 classes: 1=identified/deported; ...,ANES2024,PRE: U.S. GOVERNMENT POLICY TOWARD UNAUTHORIZE...,...,NaN,ANES Collapse to 3 classes: 1=identified/depor...,NaN,NaN,GOOD,True,1.0,1.0,2.0,Belief_opinion_on_what_should_be_done_with_ill...
4,OPINION ON WHETHER DIVORCE IN THE USA SHOULD B...,Likert,Ordinal,ANES2020,POST: FEWER PROBLEMS IF THERE WAS MORE EMPHASI...,V202265,NaN,Reverse 1-5 so higher=more agreement with trad...,NaN,NaN,...,"recode so 2 is the right level as it is, 3 is ...",NaN,NaN,"GSS 2024 only exact match, no state-level matc...",NaN,True,1.0,0.0,1.0,Belief_opinion_on_whether_divorce_in_the_usa_s...


In [6]:
title_mapper[["2024_match_id_clean","cleaned_feature"]].head()

,2024_match_id_clean,cleaned_feature
0,IMMIGPO_IMMLEVEL,Belief_opinion_on_the_number_of_immigrants_in_...
1,CSES6_IMMCRIME,Belief_opinion_on_the_statement_immigrants_inc...
2,IMMIGPO_IMMJOBS,Belief_opinion_on_the_statement_immigrants_tak...
3,IMMIG_IMMPOL,Belief_opinion_on_what_should_be_done_with_ill...
4,NaN,Belief_opinion_on_whether_divorce_in_the_usa_s...


In [9]:
title_mapper_features = title_mapper['cleaned_feature'].astype(str).str.strip()
feature_columns = pd.Index(df.columns.astype(str)).str.strip()

is_match = title_mapper_features.isin(feature_columns)

n_match = int(is_match.sum())
n_not_match = int((~is_match).sum())

print(f"Total rows checked: {len(title_mapper_features)}")
print(f"Matched: {n_match}")
print(f"Not matched: {n_not_match}")

if n_not_match > 0:
    print("\nUnmatched phrases (with counts):")
    print(title_mapper_features[~is_match].value_counts())

Total rows checked: 38
Matched: 38
Not matched: 0


In [ ]:
def build_title(row):
    gss = row['GSS 2024 benchmark']
    match_id = row['2024_match_id_clean']
    match_dataset = row['2024 Match Dataset']

    if pd.notna(gss) and pd.notna(match_id):
        return f"GSS -- {gss} / {match_dataset} -- {match_id}"
    if pd.isna(gss) and pd.notna(match_id):
        return f"{match_dataset} -- {match_id}"
    if pd.notna(gss) and pd.isna(match_id):
        return f"GSS -- {gss}"

    print("Error row (both GSS 2024 benchmark and 2024_match_id_clean are null):")
    print(row["cleaned_feature"])
    return None

title_mapper['title'] = title_mapper.apply(build_title, axis=1)
title_mapper[['GSS 2024 benchmark', '2024 Match Dataset', '2024_match_id_clean', 'title']].head()

Error row (both GSS 2024 benchmark and 2024_match_id_clean are null):
Belief_opinion_on_declining_share_of_white_people_in_america_Symbol
Error row (both GSS 2024 benchmark and 2024_match_id_clean are null):
Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol
Error row (both GSS 2024 benchmark and 2024_match_id_clean are null):
Belief_belief_on_likelihood_that_social_media_companies_are_intentionally_censoring_political_views_they_object_to_Symbol
Error row (both GSS 2024 benchmark and 2024_match_id_clean are null):
Belief_opinion_on_increasing_the_production_availability_and_usage_of_energy_from_solar_power_or_wind_turbine_farms_Symbol
Error row (both GSS 2024 benchmark and 2024_match_id_clean are null):
Belief_opinion_on_increasing_the_production_availability_and_usage_of_energy_from_nuclear_power_plants_Symbol
Error row (both GSS 2024 benchmark and 2024_match_id_clean are null):
Belief_level_of_concern_about_the_possibility_of_ukraine_being_defeated_and_taken_over_by_rus

,GSS 2024 benchmark,2024 Match Dataset,2024_match_id_clean,title,cleaned_feature
0,LETIN1A,ANES2024,IMMIGPO_IMMLEVEL,GSS -- LETIN1A / ANES2024 -- IMMIGPO_IMMLEVEL,Belief_opinion_on_the_number_of_immigrants_in_...
1,IMMCRIME1,ANES2024,CSES6_IMMCRIME,GSS -- IMMCRIME1 / ANES2024 -- CSES6_IMMCRIME,Belief_opinion_on_the_statement_immigrants_inc...
2,IMMJOBS1,ANES2024,IMMIGPO_IMMJOBS,GSS -- IMMJOBS1 / ANES2024 -- IMMIGPO_IMMJOBS,Belief_opinion_on_the_statement_immigrants_tak...
3,IMMFATE,ANES2024,IMMIG_IMMPOL,GSS -- IMMFATE / ANES2024 -- IMMIG_IMMPOL,Belief_opinion_on_what_should_be_done_with_ill...
4,DIVLAW,NaN,NaN,GSS -- DIVLAW,Belief_opinion_on_whether_divorce_in_the_usa_s...


In [11]:
title_mapper.to_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/title_mapper_with_titles.csv", index=False)

In [15]:
title_mapper_features = title_mapper['cleaned_feature'].astype(str).str.strip()
title_mapper_features[0]

'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol'